# Day 01：张量 —— 深度学习世界的通用货币

> ❄️ 第一周 · AI 的初春与寒冬 · 第 1 天

在建造感知机之前，我们需要先认识它的砖块——**张量（Tensor）**。

张量是深度学习世界的「通用货币」。不管你是 ChatGPT、自动驾驶、还是 AlphaGo，它们底层处理的数据，最终都会被拆解成一个个张量。

**今天的任务**：从零认识张量，掌握后续所有课程都需要的基础操作。

---

## 1. 张量是什么？—— 高维数组的超能力

如果你用过 NumPy，张量本质上就是一个**多维数组**。但它比普通数组多了两个超能力：

| 维度 | 名字 | 生活中的例子 |
|------|------|-------------|
| 0维 | 标量 (Scalar) | 一个数字：体温 36.5°C |
| 1维 | 向量 (Vector) | 一排数字：今天的 [气温, 湿度, 风速] |
| 2维 | 矩阵 (Matrix) | 一张表格：Excel 里的学生成绩表 |
| 3维+ | 高阶张量 | 一张彩色图片 [高度, 宽度, RGB三通道] |

让我们从创建最简单的张量开始。

In [1]:
import torch

# 0维张量：一个标量
body_temperature = torch.tensor(36.5)
print(f"体温 (0维): {body_temperature}, 形状: {body_temperature.shape}")

# 1维张量：一个向量 —— 今天的天气 [气温, 湿度, 风速]
weather_today = torch.tensor([28.0, 65.0, 12.0])
print(f"天气 (1维): {weather_today}, 形状: {weather_today.shape}")

# 2维张量：一个矩阵 —— 3个学生、2门课的成绩
#          数学  英语
grades = torch.tensor([
    [90.0, 85.0],   # 小明
    [78.0, 92.0],   # 小红
    [88.0, 76.0],   # 小刚
])
print(f"成绩 (2维):\n{grades}")
print(f"形状: {grades.shape}  →  {grades.shape[0]} 个学生, {grades.shape[1]} 门课")

体温 (0维): 36.5, 形状: torch.Size([])
天气 (1维): tensor([28., 65., 12.]), 形状: torch.Size([3])
成绩 (2维):
tensor([[90., 85.],
        [78., 92.],
        [88., 76.]])
形状: torch.Size([3, 2])  →  3 个学生, 2 门课


### 小练习

试着创建一个 1 维张量，表示你今天的步数、卡路里消耗和睡眠时长。

In [2]:
import torch
# 在这里写下你的代码
# my_health = torch.tensor([...])


---

## 2. 张量的创建方式

除了从 Python 列表手动创建，PyTorch 还提供了很多快捷方式。在神经网络中，我们经常需要初始化大量参数。

In [3]:
import torch
# 全零张量 —— 相当于一张白纸
zeros = torch.zeros(3, 4)
print(f"全零 (3x4):\n{zeros}\n")

# 全一张量
ones = torch.ones(2, 3)
print(f"全一 (2x3):\n{ones}\n")

# 随机张量 —— 神经网络初始化参数的常用方式
random_normal = torch.randn(3, 3)   # 标准正态分布（均值0，标准差1）
print(f"随机正态 (3x3):\n{random_normal}")

# 为什么用 randn 而不是 rand（均匀分布）？
# 正态分布让初始权重集中在 0 附近，有助于后续的梯度计算保持稳定
# 这是深度学习训练中一个非常重要的初始化技巧

全零 (3x4):
tensor([[0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.]])

全一 (2x3):
tensor([[1., 1., 1.],
        [1., 1., 1.]])

随机正态 (3x3):
tensor([[-0.4123,  0.0921,  0.9692],
        [ 1.0704,  0.1918,  1.3235],
        [-1.3861,  0.6465,  0.6600]])


---

## 3. 张量运算 —— 神经网络的计算基础

神经网络的本质，就是**对张量做一系列数学运算**。接下来我们拆解最核心的几种。

### 3.1 逐元素运算

两个形状相同的张量，对应位置的元素直接做加减乘除。

In [4]:
import torch
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([10.0, 20.0, 30.0])

print(f"a + b = {a + b}")        # [11, 22, 33]
print(f"a * b = {a * b}")        # [10, 40, 90]  ← 逐元素相乘，不是矩阵乘法！
print(f"a ** 2 = {a ** 2}")      # [1, 4, 9]    ← 每个元素自己平方

a + b = tensor([11., 22., 33.])
a * b = tensor([10., 40., 90.])
a ** 2 = tensor([1., 4., 9.])


### 3.2 矩阵乘法 —— 感知机的核心运算

这是今天最重要的运算。感知机的前向传播就是一步矩阵乘法：`Z = X @ W + b`。

矩阵乘法的规则：
- A 的**列数**必须等于 B 的**行数**
- 结果的形状 = A 的行数 × B 的列数

```text
[3x2]  @  [2x1]  =  [3x1]
 ↑         ↑          ↑
3个学生  2个权重    3个学生各自的得分
```

In [5]:
import torch
# 场景：3个学生，每人有2门课成绩
#       数学  英语
scores = torch.tensor([
    [90.0, 85.0],   # 小明
    [78.0, 92.0],   # 小红
    [88.0, 76.0],   # 小刚
])  # 形状: [3, 2]

# 每门课的权重（数学更重要）
weights = torch.tensor([
    [0.7],   # 数学权重
    [0.3],   # 英语权重
])  # 形状: [2, 1]

# 矩阵乘法：每个学生的加权总分
final_scores = torch.matmul(scores, weights)  # 或者用 @ 运算符: scores @ weights
print(f"加权总分:\n{final_scores}")
print(f"形状: {final_scores.shape}")  # [3, 1] —— 3个学生，每人1个总分

# 小明: 90*0.7 + 85*0.3 = 63 + 25.5 = 88.5
print(f"\n手动验证小明: {90*0.7 + 85*0.3}")

加权总分:
tensor([[88.5000],
        [82.2000],
        [84.4000]])
形状: torch.Size([3, 1])

手动验证小明: 88.5


### 3.3 广播机制 —— 不同形状也能运算

当两个张量形状不完全一致时，PyTorch 会自动「扩展」较小的那个，让它适配较大的。这叫做**广播（Broadcasting）**。

这在神经网络中非常常见，比如给每个神经元的输出加上偏置：

In [6]:
import torch
# 3个学生的加权总分 [3, 1]
final_scores = torch.tensor([[88.5], [86.2], [83.6]])

# 加一个偏置（比如考试难度调整 +5分）
bias = torch.tensor([5.0])  # 形状 [1]

# PyTorch 自动把 bias 广播成 [3, 1]，然后逐元素相加
adjusted = final_scores + bias
print(f"调整后:\n{adjusted}")

# 这就是感知机公式 Z = X@W + b 中 "+ b" 的工作原理！
# 无论 X@W 有多少行（多少个样本），b 都能自动适配

调整后:
tensor([[93.5000],
        [91.2000],
        [88.6000]])


---

## 4. 张量形状变换

在神经网络中，数据经常需要在不同层之间变换形状。

In [7]:
import torch
# 转置 —— 行变列，列变行
m = torch.tensor([[1.0, 2.0],
                   [3.0, 4.0],
                   [5.0, 6.0]])  # [3, 2]

print(f"原矩阵 [3,2]:\n{m}")
print(f"转置后 [2,3]:\n{m.T}")

# reshape —— 重塑形状，总元素数不变
flat = torch.tensor([1.0, 2.0, 3.0, 4.0, 5.0, 6.0])  # [6]
reshaped = flat.reshape(2, 3)  # [2, 3]
print(f"\nreshape [6] -> [2,3]:\n{reshaped}")

# squeeze / unsqueeze —— 去掉/增加维度为1的轴
col = torch.tensor([[1.0], [2.0], [3.0]])  # [3, 1]
print(f"\nsqueeze 前: {col.shape}")
print(f"squeeze 后: {col.squeeze().shape}")  # [3]

原矩阵 [3,2]:
tensor([[1., 2.],
        [3., 4.],
        [5., 6.]])
转置后 [2,3]:
tensor([[1., 3., 5.],
        [2., 4., 6.]])

reshape [6] -> [2,3]:
tensor([[1., 2., 3.],
        [4., 5., 6.]])

squeeze 前: torch.Size([3, 1])
squeeze 后: torch.Size([3])


---

## 5. 综合练习：用张量模拟一次「感知机计算」

虽然我们还没正式构建感知机，但你现在已经有能力用纯张量运算模拟它的核心计算了！

感知机的公式就是：`输出 = 阶跃(输入 @ 权重 + 偏置)`

让我们用今天的知识，手动走一遍。

In [8]:
import torch
# 场景：判断今天是否去打球
# 输入: [天气晴朗(0/1), 朋友叫了(0/1)]
X = torch.tensor([[0.0, 1.0],   # 雨天，朋友叫了
                   [1.0, 0.0],   # 晴天，朋友没叫
                   [1.0, 1.0]])  # 晴天，朋友叫了

# 权重: 天气权重5.0（很重要），朋友权重1.0（不那么重要）
W = torch.tensor([[5.0],
                   [1.0]])  # 形状 [2, 1]

# 偏置: -4.0（你是个宅男，门槛比较高）
b = torch.tensor([-4.0])

# 第一步：加权求和
z = torch.matmul(X, W) + b
print(f"加权求和结果 Z:\n{z}")

# 第二步：阶跃函数（>0 输出1，否则0）
decisions = (z > 0).float()
print(f"\n最终决定:\n{decisions}")

# 解读：
# [0,1]: 0*5 + 1*1 - 4 = -3 <= 0 -> 不去（雨天打败一切）
# [1,0]: 1*5 + 0*1 - 4 = 1  >  0 -> 去！（晴天就够了）
# [1,1]: 1*5 + 1*1 - 4 = 2  >  0 -> 去！（完美日）

加权求和结果 Z:
tensor([[-3.],
        [ 1.],
        [ 2.]])

最终决定:
tensor([[0.],
        [1.],
        [1.]])


---

## 今日总结

你已经掌握了深度学习最底层的「砖块」：

| 概念 | 在神经网络中的角色 |
|------|-------------------|
| 张量 | 承载一切数据的容器 |
| 矩阵乘法 @ | 感知机的核心计算：加权求和 |
| 广播 | 偏置自动适配任意数量的样本 |
| randn 初始化 | 神经网络参数的随机起点 |

**明天预告**：我们正式创建感知机的「空壳」——从生物神经元出发，理解为什么科学家要用数学模拟大脑。